# Analysis walkthrough - *Self-monitoring in perceptual decisions by humans and machines*

End-to-end run of every analysis in the manuscript, in the order the paper presents them.

| Section | Manuscript |
|---|---|
| 1. Simulation with graded bias correction | Figure 2 |
| 2. Experiment 1 - humans | Figure 3 |
| 3. Experiment 2 - speed vs accuracy focus | Figure 4 |
| 4. ANNs - standard and metacognitive readouts | Figures 5, 6 |
| 5. Supplementary analyses | Supplement |

Figure 1 is a task schematic and is not generated from data.

Sections 4 and 5 need the ANN result CSVs produced by `scripts_ann/test_metacognitive.py` (see the
README). Sections 1-3 run from the repository as cloned plus the OSF behavioural CSVs.

## Section 0 - Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from figures.style import use_publication_style
use_publication_style()

DATA = "../data/human_data"
MODEL_DATA = "../data/model_data/meta"      # produced by scripts_ann/test_metacognitive.py

## Section 1 - Simulation with graded bias correction (Figure 2)

Confidence evidence is `x + beta - alpha*beta`. Because `beta` always enters the decision
variable, choices and accuracy are **identical across the whole sweep** - only the confidence
readout changes, which is what isolates the effect of correction.

Reduce `n_subj` for a quick pass; the reported figure uses 200 observers and `alpha_step=0.1`.

In [ ]:
from scripts_analysis import simulation as sim

results = sim.run_alpha_sweep(k_list=(4, 8), n_subj=60, n_trial=400,
                              alphas=np.round(np.arange(0, 1.01, 0.25), 2), seed=1)
sim.print_summary(results)
sim.plot_alpha_sweep(results)
plt.show()

The accuracy-controlled FAR coefficient starts **positive** (bias-blind) and ends **negative**
(bias-aware), and metacognitive sensitivity Phi rises with alpha. That sign flip is the signature
the empirical analyses then look for.

## Section 2 - Experiment 1, humans (Figure 3)

Each condition is fit separately: they differ in the number of alternatives. The regression is
nested (bias -> + accuracy -> + RT); the mediation uses random `a`, `b` and `c'` slopes.

In [ ]:
from scripts_analysis.aggregate import build_human_frame
from scripts_analysis.regression import get_mixed_model_coefficients_random
from scripts_analysis.mediation import run_mediation, summarize_mediation, diagnose

CONDITIONS = {"4-choice": "Experiment1_4_choice", "8-choice": "Experiment1_8_choice"}

human_frames, human_regression = {}, {}
for label, stem in CONDITIONS.items():
    df = build_human_frame(pd.read_csv(f"{DATA}/{stem}_aggregated.csv"))
    human_frames[label] = df
    coeffs, models = get_mixed_model_coefficients_random(
        df, dependent_var="confidence_z",
        regressors=["FAR_z", "accuracy_z", "rt_z"], target_regressor="FAR_z")
    human_regression[label] = (coeffs, models)
    print(f"{label}: FAR beta (full model) = {models[2].params['FAR_z']:+.4f}")

In [ ]:
# Bayesian mediation. A few minutes per condition; `diagnose` audits convergence.
human_mediation = {}
for label, df in human_frames.items():
    trace = run_mediation(df, predictor="FAR_z")
    summary = summarize_mediation(trace)
    diagnose(trace, summary, label=label)
    human_mediation[label] = summary
    print(summary[["parameter", "mean", "hdi_2.5%", "hdi_97.5%", "stars"]].to_string(index=False))

`c_prime` is the direct effect. A negative value means confidence is discounted for alternatives
the participant over-selects, once the accuracy route is accounted for.

## Section 3 - Experiment 2, speed versus accuracy focus (Figure 4)

Both conditions are fit **together** with condition effect-coded (accuracy = +0.5, speed = -0.5),
so the Bias x Condition interaction and the accuracy-minus-speed difference in the direct effect
(`mod_direct`) each get their own estimate rather than being compared across two separate fits.

In [ ]:
from scripts_analysis.aggregate import build_combined_exp2_frame
from scripts_analysis.regression import get_moderated_regression, add_effect_coded_condition
from scripts_analysis.mediation import run_moderated_mediation, summarize_moderated_mediation

acc = pd.read_csv(f"{DATA}/Experiment2_accuracy_aggregated.csv")
spd = pd.read_csv(f"{DATA}/Experiment2_speed_aggregated.csv")
exp2 = add_effect_coded_condition(build_combined_exp2_frame(acc, spd))

moderated = get_moderated_regression(exp2, bias_var="FAR_z", outcome_var="confidence_z")
print(moderated["interaction"])
print(moderated["simple_slopes"])

In [ ]:
trace2 = run_moderated_mediation(exp2, predictor="FAR_z")
summary2 = summarize_moderated_mediation(trace2)
diagnose(trace2, summary2, label="Experiment 2 moderated mediation")
print(summary2[["parameter", "mean", "hdi_2.5%", "hdi_97.5%", "stars"]].to_string(index=False))

## Section 4 - ANNs (Figures 5 and 6)

One evaluation CSV covers both figures: `conf_top2diff` is the standard, untrained readout
(Figure 5) and `conf_meta` is the learned metacognitive head (Figure 6). Because the backbone is
frozen, FAR and accuracy are identical between them - only the confidence differs.

In [ ]:
from scripts_analysis.aggregate import aggregate_ann_csv

ANN_CSVS = {"AlexNet":  f"{MODEL_DATA}/alexnet_logit_only.csv",
            "ResNet18": f"{MODEL_DATA}/resnet18_logit_only.csv",
            "VGG19":    f"{MODEL_DATA}/vgg19_logit_only.csv"}

for readout, figure in (("conf_top2diff", "Figure 5, standard"),
                        ("conf_meta", "Figure 6, metacognitive")):
    print(f"--- {figure} ---")
    for arch, path in ANN_CSVS.items():
        df = aggregate_ann_csv(path, conf_col=readout)
        coeffs, models = get_mixed_model_coefficients_random(
            df, dependent_var="confidence_z",
            regressors=["FAR_z", "accuracy_z"], target_regressor="FAR_z")  # no RT for ANNs
        print(f"  {arch:<9} FAR beta (bias + accuracy) = {models[1].params['FAR_z']:+.4f}")

In [ ]:
# Metacognitive sensitivity (Figure 6C) -- read the caveat below before interpreting.
from scripts_analysis.metacognitive_sensitivity import phi_table, phi_accuracy_relationship

table, per_instance = phi_table(
    ANN_CSVS, readouts=("conf_top2diff", "max_softmax", "conf_meta"))
display(table)

d = pd.read_csv(ANN_CSVS["AlexNet"], usecols=["instance", "correct", "conf_meta"])
_, r = phi_accuracy_relationship(d, "conf_meta")

**How to read that panel.** `conf_top2diff` and `max_softmax` are *free* readouts of the
classifier's own outputs. `conf_meta` is a *supervised correctness classifier*, trained against
`argmax(logits) == label`. Evaluation is on held-out data, so its high Phi is not leakage - but it
is near-ceiling by construction, and Phi for the trained head correlates **negatively** with
instance accuracy (the number printed above). The gap between the two families is
supervised-versus-unsupervised, not evidence that these networks are metacognitive.

## Section 5 - Supplementary analyses

Listed by what they show; supplementary figure numbers are not stable during revision.

In [ ]:
# Trial-level: FAR of the CHOSEN alternative predicts that trial's confidence.
from scripts_analysis.trial_level import human_trial_level, ann_trial_level

human_trial_level(f"{DATA}/Experiment1_8_choice.csv", n_alternatives=8, label="8-choice")
human_trial_level(f"{DATA}/Experiment1_4_choice.csv", n_alternatives=4, label="4-choice")

for readout in ("conf_top2diff", "conf_meta"):
    ann_trial_level(ANN_CSVS["AlexNet"], conf_col=readout, label=f"AlexNet [{readout}]")

In [ ]:
# Collinearity: FAR and accuracy come from the same trials, so the VIFs must be reported.
from scripts_analysis.collinearity import vif_table, ann_vif_table, summarize_vif

tables = {label: vif_table(df, label=label) for label, df in human_frames.items()}
tables["AlexNet (ANN)"] = ann_vif_table(
    aggregate_ann_csv(ANN_CSVS["AlexNet"], conf_col="conf_meta"), label="AlexNet (ANN)")
display(summarize_vif(tables))

In [ ]:
# Individual differences in response bias.
from scripts_analysis.controls import far_variability

for label, df in human_frames.items():
    far = df.pivot_table(index="subject_idx", columns="digit", values="FAR").to_numpy()
    far_variability(far, label=label)

### Guessing controls

`controls.rt_window_sweep` and `controls.correct_only` take an `aggregate_fn` that maps a
trial-level frame to the digit-level regression frame, so they stay agnostic to each dataset's
column naming. Aggregation happens **after** trimming, so FAR and accuracy are recomputed from the
retained trials rather than merely re-filtered.

In [ ]:
from scripts_analysis.controls import rt_window_sweep, correct_only
from scripts_analysis.preprocess import aggregate_trials   # see preprocess.py for the signature

trials = pd.read_csv(f"{DATA}/Experiment1_8_choice.csv")
aggregate_fn = lambda t: build_human_frame(aggregate_trials(t, condition="8choice"))

sweep = rt_window_sweep(trials, aggregate_fn)
correct = correct_only(trials, aggregate_fn)

## Section 6 - Figures

Every panel also renders standalone, without its panel letter, so a figure can be assembled by
hand. Appearance comes from `FONTS` / `LAYOUT` / `STYLE` in `figures/style.py`.

In [ ]:
from figures.main_figures import figure3_humans

# results: {condition: {'stats': [(coef, ci, p), ...], 'scatter': [...], 'paths': {...}}}
# assembled from the regression coefficients and mediation summaries computed above.
# figure3_humans(results, out_stem="../figures_out/figure3")
# figure3_humans(results, panel="A", font_scale=1.3, out_stem="../figures_out/figure3A")